# 🔬📊 Gold-Standard Datasets - Analysis

In [1]:
# Append the parent directory to sys.path
import sys
sys.path.append("../..")

In [2]:
import os
import json
import math
import pandas as pd

In [3]:
def count_files_in_directory(directory):
    """
    Counts the total number of files in a given directory and its subdirectories.
    Args:
        directory (str): The path to the directory to count files in.
    Returns:
        int: The total number of files in the directory and its subdirectories.
    """
    file_count = 0
    for _, _, files in os.walk(directory):
        file_count += len(files)
    return file_count

In [4]:
def count_extractions_per_model(dataset: str, experiment: str, data_split: str, models: list[str], base_path: str, subfolder: str | None = None) -> dict[str, int]:
    """
    Counts the number of extraction files for each model for a given dataset and experiment.
    Args:
        dataset: Dataset name (e.g., 'BC5CDR', 'PolyIE').
        experiment: Experiment folder name (e.g., 'experiment1').
        data_split: Data split name (e.g., 'test', 'train').
        models: List of model names to count extractions for.
        base_path: Root path to the extractions directory.
        subfolder: Optional intermediate subfolder between data_split and model (e.g., 'two_judges' for multi-judge experiments).
    Returns:
        dict mapping each model name to its extraction file count.
    """
    root = os.path.join(base_path, dataset, experiment, data_split)
    counts = {}
    for model in models:
        model_path = os.path.join(root, subfolder, model) if subfolder else os.path.join(root, model)
        counts[model] = count_files_in_directory(model_path) if os.path.isdir(model_path) else 0
    return counts

In [5]:
def get_extractions_dataframe(dataset: str, experiments: list[str], data_split: str, models: list[str], base_path: str, subfolders: dict[str, list[str]] | None = None) -> pd.DataFrame:
    """
    Builds a summary DataFrame of extraction counts per model across experiments. Experiments with subfolders produce one column per subfolder (e.g., 'experiment3/two_judges'). Experiments without subfolders produce a single column named after the experiment.
    Args:
        dataset: Dataset name (e.g., 'BC5CDR', 'PolyIE').
        experiments: List of experiment names to include.
        data_split: Data split name (e.g., 'test').
        models: List of model names.
        base_path: Root path to the extractions directory.
        subfolders: Optional dict mapping experiment name to a list of intermediate subfolders (e.g., {'experiment3': ['two_judges', 'three_judges'], 'experiment4': ['self-evaluation', 'cross-evaluation']}).
    Returns:
        DataFrame indexed by model with one column per experiment (or experiment/subfolder variant).
    """
    rows = []
    for experiment in experiments:
        sf_list = (subfolders or {}).get(experiment)
        if not sf_list:
            # No subfolder — single column named after the experiment
            counts = count_extractions_per_model(dataset, experiment, data_split, models, base_path, None)
            for model, count in counts.items():
                rows.append({"column": experiment, "model": model, "count": count})
        else:
            # One column per subfolder variant
            for sf in sf_list:
                counts = count_extractions_per_model(dataset, experiment, data_split, models, base_path, sf)
                for model, count in counts.items():
                    rows.append({"column": f"{experiment}/{sf}", "model": model, "count": count})
    return pd.DataFrame(rows).pivot(index="model", columns="column", values="count").fillna(0).astype(int)

In [6]:
def get_relation_statistics(dataset: str, experiments: list[str], data_split: str, model: str, base_path: str, subfolders: dict[str, list[str]] | None = None, relations_key: str = "relations", parent_key: str | None = None) -> pd.DataFrame:
    """
    For a given dataset, computes relation extraction statistics for a given model across experiments:
    total relations extracted across all documents, and min/max relations per document.
    Experiments with subfolders produce one row per subfolder (e.g., 'experiment3/two_judges').
    Args:
        dataset: Dataset name (e.g., 'BC5CDR').
        experiments: List of experiment folder names (e.g., ['experiment1', 'experiment2']).
        data_split: Data split name (e.g., 'test').
        model: Model name to evaluate (e.g., 'gpt-4o').
        base_path: Root path to the extractions directory.
        subfolders: Optional dict mapping experiment name to a list of intermediate subfolders
            (e.g., {'experiment3': ['two_judges', 'three_judges'], 'experiment4': ['self-evaluation', 'cross-evaluation']}).
        relations_key: Key that holds the list of relations. Default is 'relations'.
        parent_key: Optional key for a parent list whose items each contain relations_key
            (e.g., 'sentences' for PolyIE where relations are nested per sentence).
    Returns:
        DataFrame indexed by experiment (or experiment/subfolder) with columns: total, min_per_doc, max_per_doc.
    """
    rows = []
    for experiment in experiments:
        sf_list = (subfolders or {}).get(experiment)
        if not sf_list:
            variants = [(experiment, None)]
        else:
            variants = [(f"{experiment}/{sf}", sf) for sf in sf_list]

        for label, sf in variants:
            model_dir = os.path.join(base_path, dataset, experiment, data_split, *([] if not sf else [sf]), model)

            if not os.path.isdir(model_dir):
                rows.append({"experiment": label, "total": 0, "average_per_doc": 0, "min_per_doc": 0, "max_per_doc": 0})
                continue

            per_doc_counts = []

            for fname in os.listdir(model_dir):
                if not fname.endswith(".json"):
                    continue
                fpath = os.path.join(model_dir, fname)
                with open(fpath, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if parent_key:
                    relations = [r for item in (data.get(parent_key) or []) for r in (item.get(relations_key) or [])]
                else:
                    relations = data.get(relations_key, []) or []
                per_doc_counts.append(len(relations))
            
            rows.append({
                "experiment": label,
                "total": sum(per_doc_counts),
                "average_per_doc": round(sum(per_doc_counts) / len(per_doc_counts), 2) if per_doc_counts else 0,
                "min_per_doc": min(per_doc_counts) if per_doc_counts else 0,
                "max_per_doc": max(per_doc_counts) if per_doc_counts else 0,
            })

    return pd.DataFrame(rows).set_index("experiment")

## 🧪 PolyIE Dataset

#### ⚗️🧮 Count of Extractions per Model Type and Experiment

In [7]:
experiments = ["experiment1", "experiment2", "experiment3", "experiment4"]
data_split = "test"
models = ["gpt-4o", "claude-sonnet-4.6", "llama-3.3-70b-instruct", "mistral-large-2512", "gemma-4-26b-a4b-it", "ministral-3b-2512"]
base_path = "../../results/extractions"
subfolders = {
    "experiment3": ["two_judges", "three_judges"],
    "experiment4": ["self-evaluation", "cross-evaluation"]
}

In [8]:
dataset = "PolyIE"
df = get_extractions_dataframe(dataset, experiments, data_split, models, base_path, subfolders)
df

column,experiment1,experiment2,experiment3/three_judges,experiment3/two_judges,experiment4/cross-evaluation,experiment4/self-evaluation
model,,,,,,
claude-sonnet-4.6,14,14,14,14,14,14
gemma-4-26b-a4b-it,14,14,14,14,0,0
gpt-4o,14,14,14,14,0,0
llama-3.3-70b-instruct,14,14,14,14,14,14
ministral-3b-2512,14,14,14,14,0,0
mistral-large-2512,14,14,14,14,0,0


#### 📊 Relation Statistics

In [9]:
df_relation_stats = get_relation_statistics(dataset="PolyIE", experiments=experiments, data_split="test", model="llama-3.3-70b-instruct", base_path="../../results/extractions", subfolders=subfolders, parent_key="sentences")
df_relation_stats

,total,average_per_doc,min_per_doc,max_per_doc
experiment,,,,
experiment1,101,7.21,4,12
experiment2,113,8.07,4,16
experiment3/two_judges,203,14.50,7,20
experiment3/three_judges,193,13.79,9,21
experiment4/self-evaluation,233,16.64,6,37
experiment4/cross-evaluation,183,13.07,7,24


## 🧪 PcMSP Dataset

#### ⚗️🧮 Count of Extractions per Model Type and Experiment

In [10]:
dataset = "PcMSP"
df = get_extractions_dataframe(dataset, experiments, data_split, models, base_path, subfolders)
df

column,experiment1,experiment2,experiment3/three_judges,experiment3/two_judges,experiment4/cross-evaluation,experiment4/self-evaluation
model,,,,,,
claude-sonnet-4.6,149,149,149,149,0,0
gemma-4-26b-a4b-it,149,149,149,149,0,0
gpt-4o,149,149,149,149,0,0
llama-3.3-70b-instruct,149,149,149,149,0,0
ministral-3b-2512,149,149,0,0,0,0
mistral-large-2512,149,149,149,149,0,0


#### 📊 Relation Statistics

In [11]:
df_relation_stats = get_relation_statistics(dataset="PcMSP", experiments=experiments, data_split="test", model="claude-sonnet-4.6", base_path="../../results/extractions", subfolders=subfolders)
df_relation_stats

,total,average_per_doc,min_per_doc,max_per_doc
experiment,,,,
experiment1,1231,8.26,1,22
experiment2,1277,8.57,1,21
experiment3/two_judges,1469,9.86,1,30
experiment3/three_judges,1391,9.34,1,24
experiment4/self-evaluation,0,0.00,0,0
experiment4/cross-evaluation,0,0.00,0,0


## 🧬 BioRED Dataset

#### 🔬🧮 Count of Extractions per Model Type and Experiment

In [14]:
dataset = "BioRED"
df = get_extractions_dataframe(dataset, experiments, data_split, models, base_path, subfolders)
df

column,experiment1,experiment2,experiment3/three_judges,experiment3/two_judges,experiment4/cross-evaluation,experiment4/self-evaluation
model,,,,,,
claude-sonnet-4.6,100,100,100,100,0,0
gemma-4-26b-a4b-it,100,100,76,87,0,0
gpt-4o,100,100,100,100,0,0
llama-3.3-70b-instruct,100,100,100,100,0,0
ministral-3b-2512,100,100,0,0,0,0
mistral-large-2512,100,100,100,100,0,0


#### 📊 Relation Statistics

In [91]:
df_relation_stats = get_relation_statistics(dataset="BioRED", experiments=experiments, data_split="test", model="claude-sonnet-4.6", base_path="../../results/extractions", subfolders=subfolders)
df_relation_stats

,total,average_per_doc,min_per_doc,max_per_doc
experiment,,,,
experiment1,773,7.73,1,43
experiment2,958,9.58,2,43
experiment3/two_judges,984,9.84,2,55
experiment3/three_judges,943,9.43,2,42
experiment4/self-evaluation,0,0.00,0,0
experiment4/cross-evaluation,0,0.00,0,0


## 🧬 BC5CDR Dataset

#### 🔬🧮 Count of Extractions per Model Type and Experiment

In [15]:
dataset = "BC5CDR"
df = get_extractions_dataframe(dataset, experiments, data_split, models, base_path, subfolders)
df

column,experiment1,experiment2,experiment3/three_judges,experiment3/two_judges,experiment4/cross-evaluation,experiment4/self-evaluation
model,,,,,,
claude-sonnet-4.6,500,500,101,184,0,0
gemma-4-26b-a4b-it,500,500,0,0,0,0
gpt-4o,500,500,91,187,0,0
llama-3.3-70b-instruct,500,500,65,80,0,0
ministral-3b-2512,500,429,0,0,0,0
mistral-large-2512,500,500,0,0,0,0


#### 📊 Relation Statistics

In [90]:
df_relation_stats = get_relation_statistics(dataset="BC5CDR", experiments=experiments, data_split="test", model="claude-sonnet-4.6", base_path="../../results/extractions", subfolders=subfolders)
df_relation_stats

,total,average_per_doc,min_per_doc,max_per_doc
experiment,,,,
experiment1,1134,2.27,0,14
experiment2,1251,2.50,0,27
experiment3/two_judges,465,2.53,0,12
experiment3/three_judges,245,2.43,0,10
experiment4/self-evaluation,0,0.00,0,0
experiment4/cross-evaluation,0,0.00,0,0
